## Model Training Summary and Comparison

## Overview of Trained Models

A total of **three main training setups** were explored:

1. **CAMeLBERT-CA on the original 12-class label setup**
2. **AraBERT v02 on the original 12-class label setup**
3. **AraBERT v02 on the final merged 7-class label setup**

## Common Training Pipeline Across Experiments

All experiments followed the same core pipeline:


### Poet-aware data split
The data was split by **poet**, not randomly by poem, to avoid leakage of poet style across train/validation/test sets.

### Fine-tuning
Each model was fine-tuned as a **sequence classifier** using weighted cross-entropy loss to handle class imbalance.

---

## Model 1  CAMeLBERT-CA (12 Classes)

### Why this model was used
This model was selected because it is an Arabic BERT model designed for Classical Arabic / MSA-like text, which makes it relevant for Arabic poetry.

### Training setup
- max length = 512
- weighted loss
- early stopping
- best model selected using validation macro-F1

### Validation Results
- **Accuracy:** 0.4838
- **Macro F1:** 0.3995
- **Weighted F1:** 0.4818
- **Validation Loss:** 1.8296


---

## Model 2 AraBERT v02 (12 Classes)

### Model
`aubmindlab/bert-base-arabertv02`


### Why this model was used
AraBERT is one of the strongest and most commonly used Arabic BERT models, making it a natural candidate for comparison.

### Training setup
This experiment used a corrected loading procedure where the pretrained weights were loaded manually and LayerNorm key names were fixed (`gamma/beta` → `weight/bias`).

This ensured that:
- the BERT backbone was loaded correctly
- only the classifier layer was newly initialized


### Test Results
- **Accuracy:** 0.4380
- **Macro F1:** 0.4122
- **Weighted F1:** 0.4363
- **Test Loss:** 1.7739


---

## Model 3 AraBERT v02 (Merged 7 Classes) Final Best Model

### Task setup
- Classification task
- Final **7-class merged label setup**

### Why the label space was changed
The original 12-class setup included several themes that were highly overlapping in meaning, such as:

- `غزل` and `رومنسيه`
- `حزينه`, `شوق`, `فراق`, and `عتاب`
- `هجاء` and `ذم`

To make the task more learnable and semantically cleaner these categories were merged into broader groups.

### Final 7 classes
- `دينية`
- `رثاء`
- `غزل_رومانسي`
- `مدح`
- `هجاء_ذم`
- `وجداني`
- `وطنية`

### Training setup
This model used:
- the same corrected AraBERT loading
- the same poet-aware split
- the same preprocessing pipeline
- the same weighted-loss training approach


### Test Results
- **Accuracy:** 0.5609
- **Macro F1:** 0.5026
- **Weighted F1:** 0.5655
- **Test Loss:** 1.3896


---

## Comparison Between All Trained Setups

| Model | Label Setup | Val Accuracy | Val Macro F1 | Test Accuracy | Test Macro F1 | Notes |
|------|-------------|-------------:|-------------:|--------------:|--------------:|------|
| CAMeLBERT-CA | 12 classes | 0.4838 | 0.3995 | — | — | Baseline experiment, but affected by weight-loading issue |
| AraBERT v02 | 12 classes | 0.4922 | 0.4222 | 0.4380 | 0.4122 | Better than CAMeLBERT, cleaner pretrained loading |
| AraBERT v02 | 7 merged classes | 0.6027 | 0.5460 | 0.5609 | 0.5026 | Final best model |


##  Environment Setup, Drive Mount, Data Loading & Cleaning
- Install required libraries
- Mount Google Drive for persistent saving
- Load the `arbml/ashaar` dataset
- Drop "قصيدة عامه", "قصيدة قصيره", and tiny classes
- Drop poems with < 3 verses
- Apply Arabic text normalization (strip diacritics, normalize alef, remove tatweel)
- Show final class distribution

In [ ]:

!pip -q install datasets transformers accelerate sentencepiece evaluate scikit-learn

import os, re, random
import numpy as np
import pandas as pd
from datasets import load_dataset

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

#  Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/arabic_poetry_topic_model"
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

# Load dataset
print("\nLoading dataset...")
dataset = load_dataset("arbml/ashaar", split="train")
df = dataset.to_pandas()
print(f"Raw dataset: {len(df):,} poems")

#  Keep only labeled rows
df = df[df["poem theme"].notna()].copy()
print(f"After keeping labeled rows: {len(df):,}")

#  Build poem text from verses
def build_poem_text(verses):
    if isinstance(verses, (list, np.ndarray)):
        items = [str(v).strip() for v in verses if str(v).strip()]
        # Pairwise reconstruction (shatr + ajuz)
        if len(items) >= 2:
            lines = []
            for i in range(0, len(items) - 1, 2):
                lines.append(items[i] + " " + items[i + 1])
            if len(items) % 2 == 1:
                lines.append(items[-1])
            return "\n".join(lines)
        return "\n".join(items)
    return ""

df["poem_text_raw"] = df["poem verses"].apply(build_poem_text)
df["num_verses"] = df["poem verses"].apply(
    lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0
)

#  Arabic normalization
ARABIC_DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]')

def normalize_arabic(text):
    if not text:
        return ""
    text = str(text)
    text = text.replace("ـ", "")                    # tatweel
    text = ARABIC_DIACRITICS.sub("", text)           # diacritics
    text = re.sub(r'[أإآٱ]', 'ا', text)             # alef variants
    text = text.replace("ة", "ه")                    # taa marbuta
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)  # non-Arabic punctuation
    text = re.sub(r'\s+', ' ', text).strip()         # whitespace
    return text

df["poem_text"] = df["poem_text_raw"].apply(normalize_arabic)
df["word_count"] = df["poem_text"].apply(lambda x: len(x.split()))

#  Filtering
# 1) Drop "قصيدة عامه" and "قصيدة قصيره"
exclude_themes = ["قصيدة عامه", "قصيدة قصيره"]
before = len(df)
df = df[~df["poem theme"].isin(exclude_themes)].reset_index(drop=True)
print(f"After dropping عامه + قصيره: {len(df):,} (removed {before - len(df):,})")

# 2) Drop very short poems (< 3 verses = < 6 items typically)
before = len(df)
df = df[df["num_verses"] >= 6].reset_index(drop=True)  # 6 items = 3 full lines
print(f"After dropping < 3 lines: {len(df):,} (removed {before - len(df):,})")

# 3) Drop tiny classes (< 50 poems)
theme_counts = df["poem theme"].value_counts()
tiny_themes = theme_counts[theme_counts < 50].index.tolist()
if tiny_themes:
    before = len(df)
    df = df[~df["poem theme"].isin(tiny_themes)].reset_index(drop=True)
    print(f"Dropped tiny classes {tiny_themes}: removed {before - len(df):,}")

#  Final distribution
print(f"\n{'='*60}")
print(f"FINAL DATASET: {len(df):,} poems | {df['poem theme'].nunique()} classes")
print(f"Unique poets: {df['poet name'].nunique()}")
print(f"{'='*60}")
final_dist = df["poem theme"].value_counts()
for theme, count in final_dist.items():
    pct = count / len(df) * 100
    print(f"  {theme:25s} → {count:>5,} ({pct:5.1f}%)")

#  Save cleaned data
df.to_pickle(os.path.join(SAVE_DIR, "df_cleaned.pkl"))
print(f"\nCleaned data saved to Drive ✓")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.6 MB/s eta 0:00:00
Mounted at /content/drive
Save directory: /content/drive/MyDrive/arabic_poetry_topic_model

Loading dataset...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/151M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/254630 [00:00<?, ? examples/s]

Raw dataset: 254,630 poems
After keeping labeled rows: 67,520
After dropping عامه + قصيره: 20,998 (removed 46,522)
After dropping < 3 lines: 18,425 (removed 2,573)
Dropped tiny classes ['قصيدة الاناشيد', 'قصيدة سياسية', 'قصيدة المعلقات', 'قصيدة اعتذار']: removed 55

FINAL DATASET: 18,370 poems | 12 classes
Unique poets: 331
  قصيدة مدح                 → 4,495 ( 24.5%)
  قصيدة رومنسيه             → 3,733 ( 20.3%)
  قصيدة حزينه               → 2,100 ( 11.4%)
  قصيدة عتاب                → 1,822 (  9.9%)
  قصيدة هجاء                → 1,281 (  7.0%)
  قصيدة غزل                 → 1,152 (  6.3%)
  قصيدة دينية               → 1,077 (  5.9%)
  قصيدة رثاء                →   813 (  4.4%)
  قصيدة شوق                 →   803 (  4.4%)
  قصيدة فراق                →   492 (  2.7%)
  قصيدة وطنيه               →   306 (  1.7%)
  قصيدة ذم                  →   296 (  1.6%)

Cleaned data saved to Drive ✓


## Poet-Aware Train/Val/Test Split + Class Weights
- Split poets (not poems) into 80/10/10
- Assign poems to splits based on their poet
- Verify no poet leaks across splits
- Compute class weights for imbalanced loss
- Save split info to Drive

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder

SEED = 42

#  Encode labels
le = LabelEncoder()
df["label"] = le.fit_transform(df["poem theme"])
label_names = list(le.classes_)
num_labels = len(label_names)
id2label = {i: label_names[i] for i in range(num_labels)}
label2id = {v: k for k, v in id2label.items()}

print(f"Number of classes: {num_labels}")
for i, name in id2label.items():
    print(f"  {i} → {name}")

#  Poet-level split
poets = sorted(df["poet name"].dropna().unique())
print(f"\nTotal unique poets: {len(poets)}")

train_poets, temp_poets = train_test_split(poets, test_size=0.2, random_state=SEED)
val_poets, test_poets = train_test_split(temp_poets, test_size=0.5, random_state=SEED)

print(f"Train poets: {len(train_poets)} | Val poets: {len(val_poets)} | Test poets: {len(test_poets)}")

# Assign splits
def assign_split(poet_name):
    if poet_name in set(train_poets):
        return "train"
    elif poet_name in set(val_poets):
        return "val"
    else:
        return "test"

df["split"] = df["poet name"].apply(assign_split)

#  Verify no leakage
train_set = set(df[df["split"]=="train"]["poet name"])
val_set = set(df[df["split"]=="val"]["poet name"])
test_set = set(df[df["split"]=="test"]["poet name"])
assert len(train_set & val_set) == 0, "LEAK: train-val overlap!"
assert len(train_set & test_set) == 0, "LEAK: train-test overlap!"
assert len(val_set & test_set) == 0, "LEAK: val-test overlap!"
print("✓ No poet leakage across splits")

#  Split sizes
print(f"\nSplit sizes:")
for sp in ["train", "val", "test"]:
    n = len(df[df["split"]==sp])
    print(f"  {sp:6s}: {n:>6,} poems")

#  Class distribution per split
print(f"\nClass distribution per split:")
ct = pd.crosstab(df["poem theme"], df["split"])
ct = ct[["train", "val", "test"]]
print(ct.to_string())

#  Compute class weights
train_labels = df[df["split"]=="train"]["label"].values
class_weights = compute_class_weight("balanced", classes=np.arange(num_labels), y=train_labels)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print(f"\nClass weights (balanced):")
for i in range(num_labels):
    print(f"  {id2label[i]:25s} → {class_weights[i]:.3f}")

#  Save to Drive
import pickle

split_info = {
    "label_encoder": le,
    "id2label": id2label,
    "label2id": label2id,
    "num_labels": num_labels,
    "train_poets": train_poets,
    "val_poets": val_poets,
    "test_poets": test_poets,
    "class_weights": class_weights,
}

with open(os.path.join(SAVE_DIR, "split_info.pkl"), "wb") as f:
    pickle.dump(split_info, f)

df.to_pickle(os.path.join(SAVE_DIR, "df_with_splits.pkl"))
print(f"\nSplit info + data saved to Drive ✓")

Number of classes: 12
  0 → قصيدة حزينه
  1 → قصيدة دينية
  2 → قصيدة ذم
  3 → قصيدة رثاء
  4 → قصيدة رومنسيه
  5 → قصيدة شوق
  6 → قصيدة عتاب
  7 → قصيدة غزل
  8 → قصيدة فراق
  9 → قصيدة مدح
  10 → قصيدة هجاء
  11 → قصيدة وطنيه

Total unique poets: 331
Train poets: 264 | Val poets: 33 | Test poets: 34
✓ No poet leakage across splits

Split sizes:
  train : 14,555 poems
  val   :    959 poems
  test  :  2,856 poems

Class distribution per split:
split          train  val  test
poem theme                     
قصيدة حزينه     1607  113   380
قصيدة دينية      913   68    96
قصيدة ذم         244    7    45
قصيدة رثاء       620   51   142
قصيدة رومنسيه   2877  257   599
قصيدة شوق        636   44   123
قصيدة عتاب      1287   98   437
قصيدة غزل        929   52   171
قصيدة فراق       355   21   116
قصيدة مدح       3835  193   467
قصيدة هجاء      1030   39   212
قصيدة وطنيه      222   16    68

Class weights (balanced):
  قصيدة حزينه               → 0.755
  قصيدة دينية               → 1.328
  ق

## Tokenize with CAMeLBERT-CA (Classical Arabic) + Build HF Datasets
- Load `CAMeL-Lab/bert-base-arabic-camelbert-ca` tokenizer
- Tokenize normalized poem text with max_length=512
- Build HuggingFace DatasetDict for train/val/test
- Audit token length coverage at 512

In [ ]:
from datasets import Dataset, DatasetDict
from transformers import AutoTokenizer

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-ca"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer: {MODEL_NAME}")
print(f"Max length: {MAX_LENGTH}")
print(f"Vocab size: {tokenizer.vocab_size}")

#  Quick token length audit (before truncation)
print("\nToken length audit (no truncation)...")
sample = df.sample(min(3000, len(df)), random_state=SEED)
lengths = sample["poem_text"].apply(lambda x: len(tokenizer.encode(x, add_special_tokens=True)))
print(f"  Mean:   {lengths.mean():.0f}")
print(f"  Median: {lengths.median():.0f}")
print(f"  P90:    {lengths.quantile(0.9):.0f}")
print(f"  P95:    {lengths.quantile(0.95):.0f}")
print(f"  Max:    {lengths.max()}")
pct_within = (lengths <= MAX_LENGTH).mean() * 100
print(f"  % fitting in {MAX_LENGTH} tokens: {pct_within:.1f}%")

#  Build HF datasets
def make_hf_dataset(split_name):
    split_df = df[df["split"] == split_name].copy()
    return Dataset.from_dict({
        "text": split_df["poem_text"].tolist(),
        "labels": split_df["label"].tolist(),
    })

raw_datasets = DatasetDict({
    "train": make_hf_dataset("train"),
    "val": make_hf_dataset("val"),
    "test": make_hf_dataset("test"),
})

print(f"\nRaw datasets:")
for sp in ["train", "val", "test"]:
    print(f"  {sp}: {len(raw_datasets[sp]):,}")

#  Tokenize
def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,  # dynamic padding via data collator
    )

tokenized_datasets = raw_datasets.map(
    tokenize_fn,
    batched=True,
    batch_size=1000,
    remove_columns=["text"],
    desc="Tokenizing",
)

print(f"\nTokenized datasets:")
for sp in ["train", "val", "test"]:
    print(f"  {sp}: {tokenized_datasets[sp]}")

#  Save tokenized datasets to Drive
tokenized_datasets.save_to_disk(os.path.join(SAVE_DIR, "tokenized_datasets"))
tokenizer.save_pretrained(os.path.join(SAVE_DIR, "tokenizer"))
print(f"\nTokenized datasets + tokenizer saved to Drive ✓")

config.json:   0%|          | 0.00/468 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Tokenizer: CAMeL-Lab/bert-base-arabic-camelbert-ca
Max length: 512
Vocab size: 30000

Token length audit (no truncation)...
  Mean:   293
  Median: 163
  P90:    715
  P95:    911
  Max:    11357
  % fitting in 512 tokens: 82.7%

Raw datasets:
  train: 14,555
  val: 959
  test: 2,856


...


Tokenized datasets:
  train: Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 14555
})
  val: Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 959
})
  test: Dataset({
    features: ['labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 2856
})


Saving the dataset (0/1 shards):   0%|          | 0/14555 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/959 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2856 [00:00<?, ? examples/s]


Tokenized datasets + tokenizer saved to Drive ✓


## Fine-tune CAMeLBERT-CA with Weighted Loss
- Load `CAMeL-Lab/bert-base-arabic-camelbert-ca` for sequence classification
- Custom Trainer with class-weighted CrossEntropyLoss
- 8 epochs, patience=3, lr=2e-5, batch=16 (via grad accum)
- Evaluate every epoch, save best by macro F1
- Save best model to Drive

In [ ]:
import os, torch
import numpy as np
import evaluate
from transformers import (
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

MODEL_NAME = "CAMeL-Lab/bert-base-arabic-camelbert-ca"
OUTPUT_DIR = "/content/camelbert_ca_poetry_topic"
BEST_MODEL_DRIVE = os.path.join(SAVE_DIR, "best_model_camelbert_ca")

#  Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    weighted_f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1}

#  Custom Trainer with class weights
class WeightedTrainer(Trainer):
    def __init__(self, class_weights_tensor=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        if class_weights_tensor is not None:
            self.class_weights = class_weights_tensor.to(self.args.device)
        else:
            self.class_weights = None

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self.class_weights is not None:
            loss_fn = torch.nn.CrossEntropyLoss(
                weight=self.class_weights.to(logits.device)
            )
        else:
            loss_fn = torch.nn.CrossEntropyLoss()
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

#  Load model
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
use_fp16 = torch.cuda.is_available()

#  Training args
num_train_samples = len(tokenized_datasets["train"])
batch_size = 8
grad_accum = 2
effective_batch = batch_size * grad_accum  # 16
steps_per_epoch = num_train_samples // effective_batch
total_steps = steps_per_epoch * 8
warmup_steps = int(0.1 * total_steps)

print(f"Effective batch size: {effective_batch}")
print(f"Steps per epoch: {steps_per_epoch}")
print(f"Total steps (8 epochs): {total_steps}")
print(f"Warmup steps: {warmup_steps}")
print(f"FP16: {use_fp16}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    save_total_limit=3,
    fp16=use_fp16,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

#  Train
trainer = WeightedTrainer(
    class_weights_tensor=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("\nStarting training...")
train_result = trainer.train()

#  Print results
print(f"\nBest checkpoint: {trainer.state.best_model_checkpoint}")

print("\nValidation metrics (best model):")
val_metrics = trainer.evaluate(tokenized_datasets["val"])
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

#  Save best model to Drive
trainer.save_model(BEST_MODEL_DRIVE)
tokenizer.save_pretrained(BEST_MODEL_DRIVE)
print(f"\nBest model saved to Drive: {BEST_MODEL_DRIVE} ✓")

pytorch_model.bin:   0%|          | 0.00/439M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: CAMeL-Lab/bert-base-arabic-camelbert-ca
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; no

Effective batch size: 16
Steps per epoch: 909
Total steps (8 epochs): 7272
Warmup steps: 727
FP16: True

Starting training...


model.safetensors:   0%|          | 0.00/439M [00:00<?, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


Best checkpoint: /content/camelbert_ca_poetry_topic/checkpoint-2730

Validation metrics (best model):


  eval_loss: 1.8296
  eval_accuracy: 0.4838
  eval_macro_f1: 0.3995
  eval_weighted_f1: 0.4818
  eval_runtime: 1.1840
  eval_samples_per_second: 809.9370
  eval_steps_per_second: 25.3370
  epoch: 6.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best model saved to Drive: /content/drive/MyDrive/arabic_poetry_topic_model/best_model_camelbert_ca ✓


## Fine-tune AraBERT v02 with Weighted Loss (Fixed Setup)
- Switch to `aubmindlab/bert-base-arabertv02` (no weight loading issues)
- Keep all improvements: max_length=512, class weights, 8 epochs, patience=3
- Re-tokenize with AraBERT tokenizer

In [ ]:
import os, torch, gc
import numpy as np
import evaluate
from collections import OrderedDict
from transformers import (
    AutoTokenizer,
    AutoConfig,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
)

gc.collect()
torch.cuda.empty_cache()

MODEL_NAME = "aubmindlab/bert-base-arabertv02"
OUTPUT_DIR = "/content/arabert_v02_fixed"
BEST_MODEL_DRIVE = os.path.join(SAVE_DIR, "best_model_arabert_v02_fixed")

#  Fix: Load weights manually and rename gamma/beta
from transformers import BertModel
import transformers

print("Loading and fixing model weights...")

# Load config
config = AutoConfig.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

# Create model with correct config
model = BertForSequenceClassification(config)

# Load pretrained weights
from huggingface_hub import hf_hub_download
import safetensors.torch

# Try loading weights
pretrained_path = hf_hub_download(repo_id=MODEL_NAME, filename="model.safetensors")
pretrained_state = safetensors.torch.load_file(pretrained_path)

# Rename gamma/beta → weight/bias
fixed_state = OrderedDict()
for key, value in pretrained_state.items():
    new_key = key
    new_key = new_key.replace("LayerNorm.gamma", "LayerNorm.weight")
    new_key = new_key.replace("LayerNorm.beta", "LayerNorm.bias")
    # Skip cls head weights (we use our own classifier)
    if new_key.startswith("cls.") or new_key == "bert.embeddings.position_ids":
        continue
    fixed_state[new_key] = value

# Load into model
missing, unexpected = model.load_state_dict(fixed_state, strict=False)
print(f"\nAfter fix:")
print(f"  Missing keys: {missing}")
print(f"  Unexpected keys: {unexpected}")

# Verify: only classifier.weight and classifier.bias should be missing
assert all("classifier" in k for k in missing), f"ERROR: Non-classifier keys missing: {missing}"
print("✓ All BERT weights loaded correctly! Only classifier head is randomly initialized.")

# Count params
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

#  Metrics
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=preds, references=labels)["accuracy"]
    macro_f1 = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    weighted_f1 = f1_metric.compute(predictions=preds, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "macro_f1": macro_f1, "weighted_f1": weighted_f1}

#  Weighted Trainer
class WeightedTrainer(Trainer):
    def __init__(self, class_weights_tensor=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights_tensor

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        if self._class_weights is not None:
            w = self._class_weights.to(logits.device)
            loss = torch.nn.CrossEntropyLoss(weight=w)(logits, labels)
        else:
            loss = torch.nn.CrossEntropyLoss()(logits, labels)
        return (loss, outputs) if return_outputs else loss

#  Training
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
use_fp16 = torch.cuda.is_available()

num_train = len(tokenized_datasets["train"])
effective_batch = 8 * 2
steps_per_epoch = num_train // effective_batch
total_steps = steps_per_epoch * 10  # more epochs now
warmup_steps = int(0.06 * total_steps)

print(f"\nSteps/epoch: {steps_per_epoch} | Total: {total_steps} | Warmup: {warmup_steps}")

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    save_total_limit=3,
    fp16=use_fp16,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
)

trainer = WeightedTrainer(
    class_weights_tensor=class_weights_tensor,
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["val"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting training (fixed weights)...")
train_result = trainer.train()

#  Results
print(f"\nBest checkpoint: {trainer.state.best_model_checkpoint}")
print("\nValidation metrics (best model):")
val_metrics = trainer.evaluate(tokenized_datasets["val"])
for k, v in val_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

#  Test evaluation
print("\nTest metrics:")
test_metrics = trainer.evaluate(tokenized_datasets["test"])
for k, v in test_metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

#  Save to Drive
trainer.save_model(BEST_MODEL_DRIVE)
tokenizer.save_pretrained(BEST_MODEL_DRIVE)
print(f"\nBest model saved to Drive: {BEST_MODEL_DRIVE} ✓")

Loading and fixing model weights...

After fix:
  Missing keys: ['classifier.weight', 'classifier.bias']
  Unexpected keys: []
✓ All BERT weights loaded correctly! Only classifier head is randomly initialized.
Total params: 135,202,572 | Trainable: 135,202,572

Steps/epoch: 909 | Total: 9090 | Warmup: 545
Starting training (fixed weights)...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,3.429796,1.657668,0.420229,0.368104,0.392224
2,3.160475,1.618216,0.467153,0.407867,0.473069
3,2.409267,1.726625,0.492179,0.422249,0.494300
4,1.888478,1.926508,0.470282,0.407060,0.477242
5,1.409910,2.086107,0.491137,0.408425,0.486384
6,0.980347,2.257080,0.493222,0.416191,0.495103


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best checkpoint: /content/arabert_v02_fixed/checkpoint-2730

Validation metrics (best model):


  eval_loss: 1.7266
  eval_accuracy: 0.4922
  eval_macro_f1: 0.4222
  eval_weighted_f1: 0.4943
  eval_runtime: 1.1911
  eval_samples_per_second: 805.1500
  eval_steps_per_second: 25.1870
  epoch: 6.0000

Test metrics:
  eval_loss: 1.7739
  eval_accuracy: 0.4380
  eval_macro_f1: 0.4122
  eval_weighted_f1: 0.4363
  eval_runtime: 3.0140
  eval_samples_per_second: 947.5800
  eval_steps_per_second: 29.8610
  epoch: 6.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best model saved to Drive: /content/drive/MyDrive/arabic_poetry_topic_model/best_model_arabert_v02_fixed ✓


## Merge to 7 Smart Classes + Retrain with Fixed Weights
- Merge overlapping classes into 7 cleaner categories
- Keep fixed weight loading
- Same strong training setup
- Compare fairly against the previous 58% benchmark

In [ ]:
import os, torch, gc
import numpy as np
import evaluate
from collections import OrderedDict
from datasets import Dataset, DatasetDict
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoConfig, AutoTokenizer,
    BertForSequenceClassification,
    DataCollatorWithPadding,
    Trainer, TrainingArguments, EarlyStoppingCallback,
)
from huggingface_hub import hf_hub_download
import safetensors.torch

gc.collect()
torch.cuda.empty_cache()

#  Merge labels
merge_map = {
    "قصيدة غزل": "غزل_رومانسي",
    "قصيدة رومنسيه": "غزل_رومانسي",
    "قصيدة هجاء": "هجاء_ذم",
    "قصيدة ذم": "هجاء_ذم",
    "قصيدة حزينه": "وجداني",
    "قصيدة شوق": "وجداني",
    "قصيدة فراق": "وجداني",
    "قصيدة عتاب": "وجداني",
    "قصيدة مدح": "مدح",
    "قصيدة رثاء": "رثاء",
    "قصيدة دينية": "دينية",
    "قصيدة وطنيه": "وطنية",
}

df["merged_theme"] = df["poem theme"].map(merge_map)
assert df["merged_theme"].isna().sum() == 0

# Re-encode
le7 = LabelEncoder()
df["label7"] = le7.fit_transform(df["merged_theme"])
label_names_7 = list(le7.classes_)
num_labels_7 = len(label_names_7)
id2label_7 = {i: label_names_7[i] for i in range(num_labels_7)}
label2id_7 = {v: k for k, v in id2label_7.items()}

print(f"Merged classes: {num_labels_7}")
for i, n in id2label_7.items():
    cnt = (df["label7"] == i).sum()
    print(f"  {i} → {n:20s} ({cnt:,})")

#  Class weights
train_labels_7 = df[df["split"]=="train"]["label7"].values
cw7 = compute_class_weight("balanced", classes=np.arange(num_labels_7), y=train_labels_7)
cw7_tensor = torch.tensor(cw7, dtype=torch.float32)
print(f"\nClass weights:")
for i in range(num_labels_7):
    print(f"  {id2label_7[i]:20s} → {cw7[i]:.3f}")

#  Build datasets
MODEL_NAME = "aubmindlab/bert-base-arabertv02"
MAX_LENGTH = 512

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def make_ds(split_name):
    s = df[df["split"]==split_name]
    return Dataset.from_dict({"text": s["poem_text"].tolist(), "labels": s["label7"].tolist()})

raw_ds = DatasetDict({
    "train": make_ds("train"), "val": make_ds("val"), "test": make_ds("test")
})

def tok_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH, padding=False)

tok_ds = raw_ds.map(tok_fn, batched=True, batch_size=1000, remove_columns=["text"], desc="Tokenizing")

for sp in ["train","val","test"]:
    print(f"{sp}: {len(tok_ds[sp]):,}")

#  Load model with FIXED weights
print("\nLoading model with fixed weights...")
config = AutoConfig.from_pretrained(MODEL_NAME, num_labels=num_labels_7, id2label=id2label_7, label2id=label2id_7)
model = BertForSequenceClassification(config)

pt_path = hf_hub_download(repo_id=MODEL_NAME, filename="model.safetensors")
pt_state = safetensors.torch.load_file(pt_path)

fixed = OrderedDict()
for k, v in pt_state.items():
    nk = k.replace("LayerNorm.gamma","LayerNorm.weight").replace("LayerNorm.beta","LayerNorm.bias")
    if nk.startswith("cls.") or nk == "bert.embeddings.position_ids":
        continue
    fixed[nk] = v

missing, unexpected = model.load_state_dict(fixed, strict=False)
assert all("classifier" in k for k in missing)
print(f"✓ Weights loaded. Missing: {missing}")

#  Metrics
acc_m = evaluate.load("accuracy")
f1_m = evaluate.load("f1")

def compute_metrics(ep):
    logits, labels = ep
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": acc_m.compute(predictions=preds, references=labels)["accuracy"],
        "macro_f1": f1_m.compute(predictions=preds, references=labels, average="macro")["f1"],
        "weighted_f1": f1_m.compute(predictions=preds, references=labels, average="weighted")["f1"],
    }

#  Weighted Trainer
class WeightedTrainer(Trainer):
    def __init__(self, class_weights_tensor=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._cw = class_weights_tensor
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        w = self._cw.to(logits.device) if self._cw is not None else None
        loss = torch.nn.CrossEntropyLoss(weight=w)(logits, labels)
        return (loss, outputs) if return_outputs else loss

#  Training
OUTPUT_DIR = "/content/arabert_7class_fixed"
BEST_DRIVE = os.path.join(SAVE_DIR, "best_model_7class_fixed")

n_train = len(tok_ds["train"])
eff_batch = 16
spe = n_train // eff_batch
total = spe * 10
warmup = int(0.06 * total)
print(f"\nSteps/epoch: {spe} | Total: {total} | Warmup: {warmup}")

args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    learning_rate=3e-5,
    weight_decay=0.01,
    warmup_steps=warmup,
    load_best_model_at_end=True,
    metric_for_best_model="eval_macro_f1",
    greater_is_better=True,
    save_total_limit=3,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=42,
)

trainer = WeightedTrainer(
    class_weights_tensor=cw7_tensor,
    model=model, args=args,
    train_dataset=tok_ds["train"], eval_dataset=tok_ds["val"],
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("Starting training (7 classes, fixed weights)...\n")
trainer.train()

#  Results
print(f"\nBest checkpoint: {trainer.state.best_model_checkpoint}")

print("\nValidation metrics:")
vm = trainer.evaluate(tok_ds["val"])
for k,v in vm.items():
    print(f"  {k}: {v:.4f}" if isinstance(v,float) else f"  {k}: {v}")

print("\nTest metrics:")
tm = trainer.evaluate(tok_ds["test"])
for k,v in tm.items():
    print(f"  {k}: {v:.4f}" if isinstance(v,float) else f"  {k}: {v}")

#  Save
trainer.save_model(BEST_DRIVE)
tokenizer.save_pretrained(BEST_DRIVE)

import pickle
with open(os.path.join(SAVE_DIR, "label_info_7class.pkl"), "wb") as f:
    pickle.dump({"le": le7, "id2label": id2label_7, "label2id": label2id_7, "merge_map": merge_map}, f)

print(f"\n✓ Everything saved to: {BEST_DRIVE}")

Merged classes: 7
  0 → دينية                (1,077)
  1 → رثاء                 (813)
  2 → غزل_رومانسي          (4,885)
  3 → مدح                  (4,495)
  4 → هجاء_ذم              (1,577)
  5 → وجداني               (5,217)
  6 → وطنية                (306)

Class weights:
  دينية                → 2.277
  رثاء                 → 3.354
  غزل_رومانسي          → 0.546
  مدح                  → 0.542
  هجاء_ذم              → 1.632
  وجداني               → 0.535
  وطنية                → 9.366


Tokenizing:   0%|          | 0/14555 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/959 [00:00<?, ? examples/s]

Tokenizing:   0%|          | 0/2856 [00:00<?, ? examples/s]

train: 14,555
val: 959
test: 2,856

Loading model with fixed weights...
✓ Weights loaded. Missing: ['classifier.weight', 'classifier.bias']

Steps/epoch: 909 | Total: 9090 | Warmup: 545
Starting training (7 classes, fixed weights)...



...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Best checkpoint: /content/arabert_7class_fixed/checkpoint-2730

📊 Validation metrics:


  eval_loss: 1.2891
  eval_accuracy: 0.6027
  eval_macro_f1: 0.5460
  eval_weighted_f1: 0.6047
  eval_runtime: 1.6887
  eval_samples_per_second: 567.8790
  eval_steps_per_second: 17.7650
  epoch: 6.0000

📊 Test metrics:
  eval_loss: 1.3896
  eval_accuracy: 0.5609
  eval_macro_f1: 0.5026
  eval_weighted_f1: 0.5655
  eval_runtime: 4.8792
  eval_samples_per_second: 585.3400
  eval_steps_per_second: 18.4460
  epoch: 6.0000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


✓ Everything saved to: /content/drive/MyDrive/arabic_poetry_topic_model/best_model_7class_fixed


##Interactive Poem Classification
- Load the best saved model
- Enter any Arabic poem and get the predicted theme

In [ ]:
import shutil, os, pickle

LSEN_DIR = "/content/drive/MyDrive/lsen"
os.makedirs(LSEN_DIR, exist_ok=True)

# 1) Save the best model + tokenizer
MODEL_DIR = os.path.join(LSEN_DIR, "best_model")
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
print(f"✓ Model saved to: {MODEL_DIR}")

# 2) Save label info
label_info = {
    "id2label": id2label_7,
    "label2id": label2id_7,
    "merge_map": merge_map,
    "num_labels": num_labels_7,
    "label_names": label_names_7,
}
with open(os.path.join(LSEN_DIR, "label_info.pkl"), "wb") as f:
    pickle.dump(label_info, f)
print(f"✓ Label info saved")

# 3) Save cleaned dataframe
df.to_pickle(os.path.join(LSEN_DIR, "df_cleaned.pkl"))
print(f"✓ Cleaned data saved")

# 4) Save class weights
torch.save(cw7_tensor, os.path.join(LSEN_DIR, "class_weights.pt"))
print(f"✓ Class weights saved")

# 5) Verify
print(f"\n Contents of {LSEN_DIR}:")
for item in os.listdir(LSEN_DIR):
    full = os.path.join(LSEN_DIR, item)
    if os.path.isdir(full):
        size = sum(os.path.getsize(os.path.join(full, f)) for f in os.listdir(full))
        print(f"  {item}/ ({size/1024/1024:.1f} MB)")
    else:
        print(f"  {item} ({os.path.getsize(full)/1024/1024:.1f} MB)")

print(f"\n كل شيء محفوظ في: {LSEN_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✓ Model saved to: /content/drive/MyDrive/lsen/best_model
✓ Label info saved
✓ Cleaned data saved
✓ Class weights saved

📁 Contents of /content/drive/MyDrive/lsen:
  📄 AraPoems_Dataset.csv (796.4 MB)
  📂 camelbert-era-classifier/ (416.9 MB)
  📂 best_model/ (517.5 MB)
  📄 label_info.pkl (0.0 MB)
  📄 df_cleaned.pkl (119.6 MB)
  📄 class_weights.pt (0.0 MB)

✅ كل شيء محفوظ في: /content/drive/MyDrive/lsen


In [1]:
#  Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

ValueError: mount failed

In [2]:
import torch, pickle, re
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification

#  Load model + tokenizer
MODEL_PATH = "/content/drive/MyDrive/lsen/best_model"
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model.eval()
if torch.cuda.is_available():
    model = model.cuda()

#  Load label info
with open("/content/drive/MyDrive/lsen/label_info.pkl", "rb") as f:
    label_info = pickle.load(f)
id2label = label_info["id2label"]

print("✓ Model loaded!")
print(f"Labels: {list(id2label.values())}")

#  Normalizer
ARABIC_DIACRITICS = re.compile(r'[\u0617-\u061A\u064B-\u0652\u0670\u06D6-\u06ED]')

def normalize_arabic(text):
    text = str(text)
    text = text.replace("ـ", "")
    text = ARABIC_DIACRITICS.sub("", text)
    text = re.sub(r'[أإآٱ]', 'ا', text)
    text = text.replace("ة", "ه")
    text = re.sub(r'[^\w\s\u0600-\u06FF]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

#  Classify function
def classify_poem(poem_text):
    clean = normalize_arabic(poem_text)
    inputs = tokenizer(clean, truncation=True, max_length=512, return_tensors="pt", padding=True)
    if torch.cuda.is_available():
        inputs = {k: v.cuda() for k, v in inputs.items()}

    with torch.no_grad():
        probs = torch.softmax(model(**inputs).logits, dim=-1)[0].cpu().numpy()

    sorted_idx = np.argsort(probs)[::-1]
    print("=" * 40)
    for i, idx in enumerate(sorted_idx[:3]):
        marker = "" if i == 0 else "  "
        print(f"  {marker} {id2label[idx]:15s} {probs[idx]*100:5.1f}%")
    print("=" * 40)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

✓ Model loaded!
Labels: ['دينية', 'رثاء', 'غزل_رومانسي', 'مدح', 'هجاء_ذم', 'وجداني', 'وطنية']


In [6]:
# ==============================
#  ضع قصيدتك هنا:
# ==============================
poem = """
يا محاكي الصم .. مجهودك سُدى
مُنصت ٍ للبكم إنصات ٍ شديد
يا مدور في الظلالات الهُدى
منتظر ابليس عن غيه يحيد
منتظر تلقى رفيق ٍ في العِدا
عاري ٍ تبغى الدفى وسط الجليد
باحث ٍ عن عز ابطى ما بتدى
باحث ٍ في هالمتاحف عن جديد
منتظر منقذ يبادر بالندا
يهزم الأعدا وحقك يستعيد
منتظر منهو يعينك بالفدا
انتظرت سنين ما جا ماتريد
عالي ٍ صوت العرب أحدث صدى
شجب وإستنكار تنديد و وعيد
حطموا هذا العدو اللي اعتدا
بالقمم أو بالخطب أو بالقصيد
يا مدور عن مثال ٍ يقتدى
لا تدور مسك في وسط الصديد
انفظ غبار الأماني ما غدا
للأماني حظ في سوق العبيد
كل وقت له نهايات ومدى
ما يجي أبطال في الوقت البليد
ما ينول المجد من يخشى الردى
في قريب الوقت والا في البعيد
كل سيف ٍ ما يجرد للصدا
كل مجد ٍ ما يجدد مايفيد
الخبر يصبح خبر بالمبتدا
والمعالي دانيه للي يريد
وكل يوم يحاك ثوب ويرتدى
وكل يوم يكفّن ويقبر فقيد
وكل مافي الكون زايل ماعدا
وجه ربك مالك الملك المجيد
يا محاكي الصم صرخاتك سُدى
لا تأذن وسط أوثان الحديد"""
classify_poem(poem)

   وجداني           74.9%
     رثاء             12.9%
     هجاء_ذم           6.1%


## How to Use the Final Trained Model as a Classifier

The final best-performing model in this notebook is a **7-class Arabic poetry theme classifier** based on:

- **Model backbone:** `aubmindlab/bert-base-arabertv02`
- **Task type:** sequence classification
- **Label setup:** merged 7-class theme classification

This means the model does **not** perform topic discovery or topic modeling.  
It performs **supervised classification**, where each input poem is assigned to one of the predefined theme labels learned during training.

---

## Final Label Space

The final classifier predicts one of the following 7 classes:

- `دينية`
- `رثاء`
- `غزل_رومانسي`
- `مدح`
- `هجاء_ذم`
- `وجداني`
- `وطنية`

These labels come from the final merged-label setup used in the best experiment.

---

## Important Rule

> Any new poem must go through the same preprocessing and tokenization steps used during training before being passed to the classifier.

If preprocessing changes, prediction quality may drop because the model was trained on normalized text, not raw text.

---

## Required Steps to Use the Classifier

### Step 1: Load the saved classifier artifacts

To use the final classifier in another script, API, or application, load:

- the saved model directory
- the saved tokenizer
- the saved label mapping (`label_info.pkl`)

These files are required because:

- the model contains the learned classifier weights
- the tokenizer defines how the Arabic text is converted into token IDs
- the label mapping converts predicted class IDs into readable class names

---

### Step 2: Receive raw input text

The classifier expects a poem or Arabic text sample as input.

This text may come from:

- a user form
- an API request
- a file
- a database

At this point, the text is still raw and should not be sent directly to the model.

---

### Step 3: Apply the same Arabic normalization used during training

Before inference, the text must be normalized using the same preprocessing logic used in the notebook.

This includes:

- removing tatweel (`ـ`)
- removing diacritics
- normalizing Alef variants (`أ`, `إ`, `آ`, `ٱ` → `ا`)
- converting `ة` to `ه`
- removing unnecessary punctuation/symbols
- removing extra spaces

This step is important because the model was trained on normalized Arabic poetry text.

---

### Step 4: Tokenize using the saved AraBERT tokenizer

Use the tokenizer saved with the final model.

Do not replace it with another tokenizer.

Use the same key inference settings:

- `truncation=True`
- `max_length=512`
- `return_tensors="pt"`

This ensures the input format matches what the classifier saw during training.

---

### Step 5: Pass the encoded input to the classifier

The tokenized output is passed to the model to produce:

- `logits`

These logits represent the raw classification scores for the 7 classes.

---

### Step 6: Convert logits into probabilities

Apply `softmax` to convert logits into probabilities.

Then you can extract:

- the predicted class
- the confidence score
- the top-k predictions if needed

---

### Step 7: Decode the predicted class ID into a label

The classifier outputs a numeric class index.

This index must be mapped back using the saved `id2label` mapping so the final output becomes a readable theme label.

---

## Full Inference Pipeline

The correct classifier pipeline is:

**Raw poem**  
→ **Arabic normalization**  
→ **AraBERT tokenizer**  
→ **Final trained classifier**  
→ **Logits**  
→ **Softmax probabilities**  
→ **Predicted class ID**  
→ **Human-readable theme label**

---

## Warnings and Best Practices

### 1. Do not skip normalization
The classifier was trained on normalized text.  
Raw text may reduce prediction quality.

### 2. Do not use another tokenizer
The saved tokenizer must always be used with the saved model.

### 3. Do not change max length unless you retrain
The final classifier was trained with:

- `max_length = 512`

Changing this at inference time is not recommended.

### 4. Long poems may be truncated
If the poem is longer than 512 tokens, it will be truncated.

This means some information may be lost for very long poems.

### 5. Predictions are constrained to the 7 learned classes
The classifier will always return one of the 7 trained labels.

It does not produce new labels and does not detect “unknown” classes automatically.

### 6. Top-3 predictions can be more useful than top-1
Some poems may carry overlapping emotional or thematic signals.

In such cases, returning the top-3 predictions is often more informative than returning only one label.

---

## Recommended Usage in Production

It is recommended to wrap the classifier into a reusable function that always performs:

1. Arabic normalization  
2. tokenization  
3. model inference  
4. probability extraction  
5. label decoding  

This helps prevent integration mistakes and keeps inference consistent with training.

---

## Summary

The final deployed model should be used as a **7-class classifier**, not as a topic model.

To use it correctly:

1. load the final saved AraBERT classifier  
2. load the saved tokenizer  
3. load the saved label mapping  
4. normalize the input text  
5. tokenize with `max_length=512`  
6. run inference  
7. convert the predicted class ID into the final label

اللي تحت دوال جاهزه للاستخدام المود في المميزات


In [ ]:
import re
import pickle
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification


def normalize_arabic(text: str) -> str:
    """
    Apply the same normalization used in the final training pipeline.
    """
    if not isinstance(text, str):
        text = str(text)

    text = text.strip()

    # Remove tatweel
    text = re.sub(r"ـ+", "", text)

    # Remove Arabic diacritics
    text = re.sub(r"[\u0617-\u061A\u064B-\u0652]", "", text)

    # Normalize Alef forms
    text = re.sub(r"[أإآٱ]", "ا", text)

    # Normalize taa marbouta
    text = re.sub(r"ة", "ه", text)

    # Remove punctuation/symbol noise
    text = re.sub(r"[^\u0600-\u06FF0-9A-Za-z\s]", " ", text)

    # Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def load_final_classifier(model_dir: str, label_info_path: str, device: str = None):
    """
    Load the final best classifier (AraBERT v02, 7 classes) and label mapping.
    """
    model_dir = Path(model_dir)
    label_info_path = Path(label_info_path)

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    model.to(device)
    model.eval()

    with open(label_info_path, "rb") as f:
        label_info = pickle.load(f)

    if "id2label" not in label_info:
        raise ValueError("label_info.pkl does not contain 'id2label'.")

    id2label = {int(k): v for k, v in label_info["id2label"].items()}

    return model, tokenizer, id2label, device


def predict_poem_theme(
    text: str,
    model,
    tokenizer,
    id2label: dict,
    device: str,
    max_length: int = 512,
    top_k: int = 3
):
    """
    Predict the theme of a poem using the final 7-class classifier.

    Steps:
    1. Normalize raw Arabic text
    2. Tokenize using the saved AraBERT tokenizer
    3. Run the classifier
    4. Convert logits to probabilities
    5. Return top-k predictions
    """
    clean_text = normalize_arabic(text)

    inputs = tokenizer(
        clean_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_length,
        padding=False
    )

    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1).squeeze(0)

    top_k = min(top_k, probs.shape[0])
    top_probs, top_ids = torch.topk(probs, k=top_k)

    results = []
    for prob, idx in zip(top_probs.tolist(), top_ids.tolist()):
        results.append({
            "label_id": idx,
            "label": id2label[idx],
            "probability": round(prob, 4)
        })

    return {
        "raw_text": text,
        "normalized_text": clean_text,
        "predicted_label": results[0]["label"],
        "confidence": results[0]["probability"],
        "top_k_predictions": results
    }

In [ ]:
MODEL_DIR = "/content/drive/MyDrive/lsen/best_model"
LABEL_INFO_PATH = "/content/drive/MyDrive/lsen/label_info.pkl"

model, tokenizer, id2label, device = load_final_classifier(
    model_dir=MODEL_DIR,
    label_info_path=LABEL_INFO_PATH
)

sample_text = """
يا وطني يا موطن الامجاد والفخر
فيك العز يبقى ما بقي الدهر
"""

result = predict_poem_theme(
    text=sample_text,
    model=model,
    tokenizer=tokenizer,
    id2label=id2label,
    device=device,
    top_k=3
)

print("Predicted label:", result["predicted_label"])
print("Confidence:", result["confidence"])
print("Top predictions:")
for item in result["top_k_predictions"]:
    print(f"- {item['label']}: {item['probability']}")